In [1]:
# glavni program
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import pandas as pd

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

def calculate_angle(point_a, point_b, point_c):
    point_a, point_b, point_c = np.array(point_a), np.array(point_b), np.array(point_c)
    vector_ba = point_a - point_b
    vector_bc = point_c - point_b
    cosine_angle = np.dot(vector_ba, vector_bc) / (np.linalg.norm(vector_ba) * np.linalg.norm(vector_bc))
    angle_in_degrees = np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))
    return angle_in_degrees

video_path = './Videos/Video.mov'  
video_capture = cv2.VideoCapture(video_path)
frames_per_second = video_capture.get(cv2.CAP_PROP_FPS)

push_off_threshold = 0.05
min_swing_frames = frames_per_second * 0.22 
min_contact_frames = 5 

all_step_metrics_storage = []
history_window_size = 10 
right_knee_angle_history = deque(maxlen=history_window_size)
left_knee_angle_history = deque(maxlen=history_window_size)
hip_height_history = deque(maxlen=50)

current_max_split = 0
leg_is_on_ground = {"Right": False, "Left": False}
ankle_y_at_contact = {"Right": None, "Left": None}
last_leg_that_landed = None
last_strike_frame_index = 0
first_strike_frame = None
previous_strike_frame = None
current_instant_cadence = 0
right_step_count = 0
left_step_count = 0
current_status_event = "CALIBRATING"

with mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7) as pose_analyzer:
    while video_capture.isOpened():
        ret, frame = video_capture.read()
        if not ret: break
        
        overlay_layer = frame.copy()
        display_frame = frame.copy()
        frame_height, frame_width, _ = frame.shape
        results = pose_analyzer.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            def get_pixel_point(idx): return np.array([int(landmarks[idx].x * frame_width), int(landmarks[idx].y * frame_height)])

            right_hip, right_knee, right_ankle = get_pixel_point(24), get_pixel_point(26), get_pixel_point(28)
            left_hip, left_knee, left_ankle = get_pixel_point(23), get_pixel_point(25), get_pixel_point(27)
            right_shoulder, right_elbow, right_wrist = get_pixel_point(12), get_pixel_point(14), get_pixel_point(16)
            left_shoulder, left_elbow, left_wrist = get_pixel_point(11), get_pixel_point(13), get_pixel_point(15)
            
            right_knee_angle = calculate_angle(right_hip, right_knee, right_ankle)
            left_knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
            right_elbow_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)
            left_elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
            
            trunk_vector = np.array([landmarks[12].x - landmarks[24].x, landmarks[12].y - landmarks[24].y])
            trunk_angle = 180 - np.degrees(np.arctan2(np.abs(trunk_vector[0]), np.abs(trunk_vector[1])))
            
            torso_height = np.abs(landmarks[24].y - landmarks[12].y) 
            mid_hip_y = (landmarks[24].y + landmarks[23].y) / 2
            hip_height_history.append(mid_hip_y)
            vertical_oscillation_percent = ((max(hip_height_history) - min(hip_height_history)) / torso_height) * 100 if torso_height > 0 else 0

            vector_right, vector_left = (right_knee - right_hip), (left_knee - left_hip)
            split_angle = np.degrees(np.arccos(np.clip(np.dot(vector_right/np.linalg.norm(vector_right), vector_left/np.linalg.norm(vector_left)), -1.0, 1.0)))
            if split_angle > current_max_split: current_max_split = split_angle

            torso_points = np.array([right_shoulder, left_shoulder, left_hip, right_hip], np.int32)
            cv2.fillPoly(overlay_layer, [torso_points], (0, 255, 0)) 
            cv2.addWeighted(overlay_layer, 0.15, display_frame, 0.85, 0, display_frame)
            cv2.polylines(display_frame, [torso_points], True, (255, 255, 255), 2)
            cv2.line(display_frame, tuple(right_shoulder), tuple(left_hip), (255, 255, 255), 1)
            cv2.line(display_frame, tuple(left_shoulder), tuple(right_hip), (255, 255, 255), 1)

            cv2.line(display_frame, tuple(right_ankle), tuple(left_ankle), (255, 0, 255), 2)
            cv2.line(display_frame, tuple(right_hip), (right_hip[0], right_hip[1]-80), (0, 0, 255), 1)

            for h, k, a, c in [(right_hip, right_knee, right_ankle, (0, 255, 0)), (left_hip, left_knee, left_ankle, (0, 255, 255))]:
                cv2.line(display_frame, tuple(h), tuple(k), c, 3)
                cv2.line(display_frame, tuple(k), tuple(a), c, 3)
            
            for s, e, w in [(right_shoulder, right_elbow, right_wrist), (left_shoulder, left_elbow, left_wrist)]:
                cv2.line(display_frame, tuple(s), tuple(e), (255, 165, 0), 3)
                cv2.line(display_frame, tuple(e), tuple(w), (255, 165, 0), 3)

            for j, ang, c in [(right_knee, right_knee_angle, (0,255,0)), (left_knee, left_knee_angle, (0,255,255)), (right_elbow, right_elbow_angle, (255,165,0))]:
                cv2.putText(display_frame, f"{int(ang)}", tuple(j + [10, -10]), 1, 1.2, c, 2)

            right_knee_angle_history.append(right_knee_angle)
            left_knee_angle_history.append(left_knee_angle)
            current_frame = video_capture.get(cv2.CAP_PROP_POS_FRAMES)

            if len(right_knee_angle_history) == history_window_size:
                middle_index = history_window_size // 2
                for side, history, ankle_y, elbow_angle in [("Right", right_knee_angle_history, landmarks[28].y, right_elbow_angle), ("Left", left_knee_angle_history, landmarks[27].y, left_elbow_angle)]:
                    if (last_leg_that_landed != side and history[middle_index] > 150 and history[middle_index] == max(history) and (current_frame - last_strike_frame_index) > min_swing_frames):
                        leg_is_on_ground[side], ankle_y_at_contact[side], last_leg_that_landed = True, ankle_y, side
                        current_status_event = f"{side.upper()} STRIKE"
                        
                        if first_strike_frame is None: first_strike_frame = current_frame
                        
                        if previous_strike_frame is not None:
                            frames_between = current_frame - previous_strike_frame
                            current_instant_cadence = (60 * frames_per_second) / frames_between
                        
                        previous_strike_frame = current_frame
                        last_strike_frame_index = current_frame
                        if side == "Right": right_step_count += 1
                        else: left_step_count += 1
                        
                        # Inicijaliziramo max_knee_during_contact s kutom u trenutku udarca
                        all_step_metrics_storage.append({
                            'side': side, 
                            'strike_knee': history[middle_index], 
                            'split': current_max_split, 
                            'trunk': trunk_angle, 
                            'cadence': current_instant_cadence, 
                            'v_osc': vertical_oscillation_percent, 
                            'elbow': elbow_angle, 
                            'start_frame': current_frame, 
                            'max_knee_during_contact': history[middle_index], 
                            'done': False
                        })
                        current_max_split = 0

                for side, current_y in [("Right", landmarks[28].y), ("Left", landmarks[27].y)]:
                    if leg_is_on_ground[side]:
                        for record in reversed(all_step_metrics_storage):
                            if record['side'] == side and not record['done']:
                                # Stalno pratimo maksimum dok je noga na tlu
                                current_angle = right_knee_angle if side == "Right" else left_knee_angle
                                if current_angle > record['max_knee_during_contact']:
                                    record['max_knee_during_contact'] = current_angle
                                
                                frames_on_ground = current_frame - record['start_frame']
                                if (ankle_y_at_contact[side] - current_y) > push_off_threshold and frames_on_ground >= min_contact_frames:
                                    leg_is_on_ground[side] = False
                                    current_status_event = f"{side.upper()} PUSH-OFF"
                                    record['gct'] = (frames_on_ground / frames_per_second) * 1000
                                    # Spremamo zabilježeni maksimum ekstenzije
                                    record['push_knee'] = record['max_knee_during_contact']
                                    record['done'] = True
                                break

            cv2.rectangle(display_frame, (0,0), (280, 100), (20,20,20), -1)
            cv2.putText(display_frame, f"STEPS: {right_step_count + left_step_count}", (15, 35), 1, 1.8, (255,255,255), 2)
            cv2.putText(display_frame, f"STATUS: {current_status_event}", (15, 75), 1, 1.2, (0, 255, 255), 1)

            for side, position, state in [("LEFT", (10, frame_height-20), leg_is_on_ground["Left"]), ("RIGHT", (frame_width-130, frame_height-20), leg_is_on_ground["Right"])]:
                color = (0, 255, 0) if state else (0, 0, 255)
                cv2.rectangle(display_frame, (position[0]-10, position[1]-40), (position[0]+120, position[1]+10), (0,0,0), -1)
                cv2.putText(display_frame, side, (position[0], position[1]-20), 1, 1.2, color, 2)
                cv2.putText(display_frame, "CONTACT" if state else "FLIGHT", (position[0], position[1]), 1, 0.9, (255,255,255), 1)

            cv2.imshow('Running analysis', display_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'): break

video_capture.release()
cv2.destroyAllWindows()

I0000 00:00:1769037169.313702 2119841 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1769037169.383492 2120039 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769037169.392793 2120039 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769037169.405137 2120043 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


In [4]:
# kreiranje tablice za dataset
import csv
import os

file_path = 'dataset.csv'
fieldnames = [
    'rep_number', 'frame_index', 'file_name', 'side',
    'cadence_value (spm)', 'gct_value (ms)', 'vert_osc_value (%)', 
    'knee_strike_angle (deg)', 'knee_push_angle (deg)', 'elbow_angle_val (deg)', 
    'leg_split_val (deg)', 'trunk_lean_value (deg)',
    'cadence_score', 'gct_score', 'vert_osc_score', 'knee_strike_score', 
    'knee_push_score', 'elbow_angle_score', 'leg_split_score', 'trunk_lean_score'
]

if not os.path.exists(file_path):
    with open(file_path, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
    print(f"Tablica {file_path} uspješno kreirana.")
else:
    print(f"Tablica {file_path} već postoji.")

Tablica dataset.csv već postoji.


In [5]:
# dodavanje u dataset
import csv
import os

def get_metric_score(metric, val):
    if metric == "cadence": return 3 if val > 175 else (2 if val > 168 else 1)
    if metric == "gct": return 3 if val < 265 else (2 if val < 283 else 1)
    
    if metric == "push_knee":
        if val > 165: return 3         
        if 155 <= val <= 165: return 2
        if 150 <= val <= 155: return 1
        return 0
    
    if metric == "strike_knee":
        if 155 <= val <= 165: return 3
        if 150 <= val <= 170: return 2
        if 145 <= val <= 175: return 1
        return 0
    
    if metric == "elbow":
        if 70 <= val <= 100: return 3
        if 60 <= val <= 115: return 2
        if 50 <= val <= 120: return 1
        return 0
    
    if metric == "v_osc":
        if val < 15.0: return 3
        if val < 18.0: return 2
        if val < 21.0: return 1
        return 0

    if metric == "trunk":
        if 160 <= val <= 180: return 3
        if 150 <= val <= 185: return 2
        if 145 <= val <= 190: return 1
        return 0

    if metric == "split":
        if 65 <= val <= 75: return 3
        if 60 <= val <= 80: return 2
        if 55 <= val <= 85: return 1
        return 0
    return 0

line_count = 0
if os.path.exists(file_path):
    with open(file_path, 'r', newline='') as f:
        line_count = sum(1 for row in csv.reader(f)) - 1

if 'all_step_metrics_storage' in locals() and all_step_metrics_storage:
    with open(file_path, 'a', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        for i, m in enumerate(all_step_metrics_storage, start=max(0, line_count) + 1):
            if not m.get('done'): continue
            
            cad = m['cadence']
            gct = m.get('gct', 0)
            v_osc = m['v_osc']
            s_knee = m['strike_knee']
            p_knee = m.get('push_knee', 0)
            elb = m['elbow']
            split = m['split']
            trunk = m['trunk']

            row = {
                'rep_number': i,
                'frame_index': int(m['start_frame']),
                'file_name': video_path,
                'side': m['side'],
                'cadence_value (spm)': round(cad, 2),
                'gct_value (ms)': int(gct),
                'vert_osc_value (%)': round(v_osc, 2),
                'knee_strike_angle (deg)': int(s_knee),
                'knee_push_angle (deg)': int(p_knee),
                'elbow_angle_val (deg)': int(elb),
                'leg_split_val (deg)': int(split),
                'trunk_lean_value (deg)': int(trunk),
                
                'cadence_score': get_metric_score("cadence", cad),
                'gct_score': get_metric_score("gct", gct),
                'vert_osc_score': get_metric_score("v_osc", v_osc),
                'knee_strike_score': get_metric_score("strike_knee", s_knee),
                'knee_push_score': get_metric_score("push_knee", p_knee),
                'elbow_angle_score': get_metric_score("elbow", elb),
                'leg_split_score': get_metric_score("split", split),
                'trunk_lean_score': get_metric_score("trunk", trunk)
            }
            writer.writerow(row)
    print(f"Podaci uspješno dodani u {file_path}")
else:
    print("all_step_metrics_storage je prazan ili ne postoji.")

Podaci uspješno dodani u dataset.csv


In [6]:
# visualization
if all_step_metrics_storage:
    final_results = []
    keys = ["cadence", "gct", "v_osc", "strike_knee", "push_knee", "trunk", "split", "elbow"]
    
    for k in keys:
        valid_vals = [step[k] for step in all_step_metrics_storage if k in step and step.get('done') == True and step[k] > 0]
        avg = np.mean(valid_vals) if valid_vals else 0
        score = get_metric_score(k, avg)
        final_results.append({"METRIC": k.upper().replace("_", " "), "AVG VALUE": round(avg, 1), "SCORE": score})

    df = pd.DataFrame(final_results)
    print("\n" + "═"*45)
    print("      GAIT PERFORMANCE SUMMARY")
    print("═"*45)
    print(df.to_string(index=False))
    print("─" * 45)
    print(f"TOTAL SCORE: {df['SCORE'].sum()} / 24")
    print("═"*45)
else:
    print("No complete gait data captured.")


═════════════════════════════════════════════
      GAIT PERFORMANCE SUMMARY
═════════════════════════════════════════════
     METRIC  AVG VALUE  SCORE
    CADENCE      187.2      3
        GCT       84.3      3
      V OSC       18.1      1
STRIKE KNEE      156.8      3
  PUSH KNEE      156.8      2
      TRUNK      178.1      3
      SPLIT       74.7      3
      ELBOW      105.8      2
─────────────────────────────────────────────
TOTAL SCORE: 20 / 24
═════════════════════════════════════════════
